In [4]:
import pandas as pd
from scipy.io import savemat
import matlab.engine

def create_segment(shift):
    n_initial_240s = 22 + shift
    n_150s = 19
    n_final_240s = 96 - n_initial_240s - n_150s
    
    segment = [120] * n_initial_240s + [0] * n_150s + [120] * n_final_240s
    final_segment = segment * 14
    return final_segment

def generate_sequence(length):
    seq = []
    value = 0
    for _ in range(length):
        seq.append(value)
        value += 1/96
        if round(value, 2) >= 14:
            value = 0
    return seq

def save_to_mat(data):
    df = pd.DataFrame(data, columns=['Value'])
    df['Sequence'] = generate_sequence(len(df))

    # Validate the data shape
    if df.shape != (1344, 2):
        raise ValueError(f"Unexpected data shape: {df.shape}. Expected (1344, 2).")
    
    # Create combined 2D list
    combined_data = df[['Sequence', 'Value']].values.tolist()
    
    # Save this combined list as a single variable in the .mat file
    data_dict_tank3 = {'KLa3_Setpoints_BSM2': combined_data}
    data_dict_tank4 = {'KLa4_Setpoints_BSM2': combined_data}
    data_dict_tank5 = {'KLa5_Setpoints_BSM2': combined_data}
    
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/KLa5_Setpoints_BSM2.mat', data_dict_tank5)
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/KLa4_Setpoints_BSM2.mat', data_dict_tank4)
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/KLa3_Setpoints_BSM2.mat', data_dict_tank3)

def run_matlab_model(iteration):
    eng = matlab.engine.start_matlab()

    # Save the 'iteration' value as a MATLAB datafile
    iteration_data = {"iteration": iteration}
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/iteration.mat', iteration_data)

    eng.cd('/Users/aya/github/WWDR/BSM1-BSM2_MATLAB/BSM2_R2019b', nargout=0)
    eng.run('run_TSaeration_DRbsm2.m', nargout=0)
    eng.quit()

# Get the number of iterations
n_segments = int(input("Enter the number of iterations: "))

shift = 0
for i in range(n_segments):
    data = create_segment(shift)
    save_to_mat(data)
    run_matlab_model(i + 1)  # Pass the current iteration count, starting from 1
    shift += 1



Error using :
Double operands interacting with int64 operands must have integer values.

Error in run_TSaeration_DRbsm2 (line 11)
outputtimes=[0:(1/96):days*96]; %Define the simulation time for dynamic influent

Error in run (line 91)
evalin('caller', strcat(script, ';'));



MatlabExecutionError: 
  File /Users/aya/github/WWDR/BSM1-BSM2_MATLAB/BSM2_R2019b/run_TSaeration_DRbsm2.m, line 11, in run_TSaeration_DRbsm2

  File /Applications/MATLAB_R2022b.app/toolbox/matlab/lang/run.m, line 91, in run
Double operands interacting with int64 operands must have integer values.


In [138]:
dftest = pd.DataFrame(test_segment, columns=['Val'])
dftest['Seq'] = testseq
# combo_dftest = df[['Seq','Val']].values.tolist()
dftest[23520:]

,Val,Seq
23520,120,245.000000
23521,120,245.010417
23522,120,245.020833
23523,120,245.031250
23524,120,245.041667
...,...,...
27547,120,286.947917
27548,120,286.958333
27549,120,286.968750
27550,120,286.979167


In [ ]:
indices = dftest.index[dftest['Val'] == 0].tolist()
# print(f"Indices where Value is 0: {indices}")
print(((max(indices)/96)-245-28))

13.635416666666686


## Working Code Below   

In [ ]:
import numpy as np
import pandas as pd
from scipy.io import savemat
import matlab.engine

def create_segment(DR_len, kla, shift):
    n_calibration = (245 + shift*14)*96
    initial_segment = [120] * n_calibration
    
    n_ininominal = 32 #22 is roughly 8:00AM
    n_DR = DR_len * 4 
    n_fnlnominal = 96 - n_ininominal - n_DR
    
    segment = [120] * n_ininominal + [kla] * n_DR + [120] * n_fnlnominal
    final_segment = segment * 14
    return initial_segment + final_segment

def generate_sequence(length):
    seq = []
    value = 0
    for _ in range(length):
        seq.append(value)
        value += 1/96
        if round(value, 2) >= length:
            value = 0
    return seq

def save_to_mat(data):
    df = pd.DataFrame(data, columns=['Value'])
    df['Sequence'] = generate_sequence(len(df))
 
    # Create combined 2D list
    combined_data = df[['Sequence', 'Value']].values.tolist()
    
    # Save this combined list as a single variable in the .mat file
    data_dict_tank3 = {'KLa3_Setpoints_BSM2': combined_data}
    data_dict_tank4 = {'KLa4_Setpoints_BSM2': combined_data}
    data_dict_tank5 = {'KLa5_Setpoints_BSM2': combined_data}
    
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/KLa5_Setpoints_BSM2.mat', data_dict_tank5)
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/KLa4_Setpoints_BSM2.mat', data_dict_tank4)
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/KLa3_Setpoints_BSM2.mat', data_dict_tank3)

def run_matlab_model(iteration):
    eng = matlab.engine.start_matlab()

    # Save the 'iteration' value as a MATLAB datafile
    iteration_data = {"iteration": np.array([iteration], dtype=np.float64)}
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/iteration.mat', iteration_data)

    eng.cd('/Users/aya/github/WWDR/BSM1-BSM2_MATLAB/BSM2_R2019b', nargout=0)
    eng.run('run_TSaeration_DRbsm2.m', nargout=0)
    eng.quit()

# Get the number of iterations
n_segments = int(input("Enter the number of iterations: "))

shift = 0
for i in range(n_segments):
    data = create_segment(5,0,shift)
    save_to_mat(data)
    run_matlab_model(i + 1)  # Pass the current iteration count, starting from 1
    shift += 1



Current iteration value is: 1
 
Running BSM2 to steady state! Solver = ode15s and Simulink model = benchmarkss
**************************************************************************
 
Steady state achieved. Initializing all state variables to steady state values.
 
Simulating BSM2 with dynamic influent (i) in open loop (Tempmodel = 1)! Solver = ode45 and Simulink model = benchmark
*****************************************************************************************************************
 
Start time for simulation (hour:min:sec) = 1  37  22
Dynamic open loop BSM2 simulation finished!
End time for simulation (hour:min:sec) = 3  31  12
 
 
***** Plant evaluation of BSM2 system initiated *****
Start time for BSM2 evaluation (hour:min:sec) = 3  31  18
 
 
Overall plant performance during time 245 to 259 days
*****************************************************
 
Effluent average concentrations based on load
---------------------------------------------
Effluent average flow rate

# Scratch work ahead

In [11]:
import numpy as np
import pandas as pd
from scipy.io import savemat
import os

def generate_time_sequence(length):
    return [i / 96 for i in range(length)]

def create_nominal_kla_sequence(days, kla_value=120, tank=3):
    total_steps = days * 96
    if tank == 5:
        kla_value = kla_value / 2
    return [kla_value] * total_steps

def create_experiment_kla_sequence(days, kla_value, DR_len):
    steps_per_day = 96
    n_ininominal = 22
    n_DR = DR_len * 4
    n_fnlnominal = steps_per_day - n_ininominal - n_DR
    day_pattern = [120] * n_ininominal + [kla_value] * n_DR + [120] * n_fnlnominal
    return day_pattern * days

def save_kla_to_mat(kla_sequence, tank, tag):
    time_seq = generate_time_sequence(len(kla_sequence))
    df = pd.DataFrame({'Sequence': time_seq, 'Value': kla_sequence})
    combined = df[['Sequence', 'Value']].values.tolist()

    var_name = f'KLa{tank}_Setpoints_BSM2'
    filename = f'BSM1-BSM2_MATLAB/BSM2_R2019b/KLa{tank}_Setpoints_BSM2_{tag}.mat'
    savemat(filename, {var_name: combined})
    print(f"Saved: {filename}")



In [12]:

# === Example Usage ===
days = 609
kla_nominal_value = 120
kla_experiment_value = 0
experiment_length = 4  # hours

for tank in [3, 4, 5]:
    # Generate sequences
    nominal_seq = create_nominal_kla_sequence(days, kla_nominal_value, tank=tank)
    experiment_seq = create_experiment_kla_sequence(days, kla_experiment_value, experiment_length)

    # Save to .mat files
    save_kla_to_mat(nominal_seq, tank=tank, tag='nominal')
    save_kla_to_mat(experiment_seq, tank=tank, tag='experiment')


Saved: BSM1-BSM2_MATLAB/BSM2_R2019b/KLa3_Setpoints_BSM2_nominal.mat
Saved: BSM1-BSM2_MATLAB/BSM2_R2019b/KLa3_Setpoints_BSM2_experiment.mat
Saved: BSM1-BSM2_MATLAB/BSM2_R2019b/KLa4_Setpoints_BSM2_nominal.mat
Saved: BSM1-BSM2_MATLAB/BSM2_R2019b/KLa4_Setpoints_BSM2_experiment.mat
Saved: BSM1-BSM2_MATLAB/BSM2_R2019b/KLa5_Setpoints_BSM2_nominal.mat
Saved: BSM1-BSM2_MATLAB/BSM2_R2019b/KLa5_Setpoints_BSM2_experiment.mat


In [1]:
import numpy as np
import pandas as pd
from scipy.io import savemat
import matlab.engine

In [2]:
def run_DR_calibration_model():
    eng = matlab.engine.start_matlab()

    eng.cd('/Users/aya/github/WWDR/BSM1-BSM2_MATLAB/BSM2_R2019b', nargout=0)
    eng.run('DR_BSM2_calibration.m', nargout=0)
    eng.quit()

In [9]:
run_DR_calibration_model()

 
Running BSM2 to steady state! Solver = ode15s and Simulink model = benchmark2 ss
**************************************************************************
 
Steady state achieved. Initializing all state variables to steady state values.
 
Simulating BSM2 with dynamic influent (i) in open loop (Tempmodel = 1)! Solver = ode45 and Simulink model = benchmark2 open loop
*****************************************************************************************************************
 
Start time for simulation (hour:min:sec) = 20   3   2
Simulation paused at t = 14.0113 days.
Simulation resumed after custom actions.
the MATLAB function has been cancelled


Operation terminated by user during DR_BSM2_calibration


In run (line 91)
evalin('caller', strcat(script, ';'));



In [ ]:
def run_DR_experimental_model(iteration):
    DRflag = 1
    if DRflag < 1:
        DRflag = 1
    eng = matlab.engine.start_matlab()

    # Save the 'iteration' value as a MATLAB datafile
    DRflag_data = {"DRflag": np.array([DRflag], dtype=np.float64)}
    iteration_data = {"iteration": np.array([iteration], dtype=np.float64)}

    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/iteration.mat', iteration_data)
    savemat('BSM1-BSM2_MATLAB/BSM2_R2019b/DRflag.mat', DRflag_data)

    DRflag = DRflag - 1

    eng.cd('/Users/aya/github/WWDR/BSM1-BSM2_MATLAB/BSM2_R2019b', nargout=0)
    eng.run('run_bsm2_iteration_workflow.m', nargout=0)
    eng.quit()

# Get the number of iterations
n_segments = int(input("Enter the number of iterations: "))

shift = 0
for i in range(n_segments):
    run_DR_experimental_model(i + 1)  # P

Current iteration value is: 1
Simulating DR Experiment BSM2 with dynamic influent (i) in open loop (Tempmodel = 1)! Solver = ode45 and Simulink model = benchmark2 open loop
*****************************************************************************************************************
 
Start time for simulation (hour:min:sec) = 19  22  52
Current iteration value is: 2
